# Najafi et al. (2020) on DANDI:000016, part 4: decoder time course with C chosen at every bin

Paper doi:10.1016/j.neuron.2019.09.045; dataset DANDI:000016 version 0.260915.2309, the only
published version (the paper cites none; data checks in `01_single_neurons_correlations.ipynb`).
Claims file `claims/najafi-2020.yaml`.

`03_learning_decoders.ipynb` tests fig3b with C chosen at bin -1 and reused at every bin, with 25
random splits. The claims file's opening comment chooses C separately at every bin, over 50
splits. This notebook repeats fig3b that way for mice 2 and 3, the two mice whose peak fell
outside the window and where the verifier asked for it (mouse 2's peak at bin 3 led bin 2 by 0.1
points). For runtime it covers the earliest bin of each session and every bin from -3 (291 ms
before the choice) to the last, the bins that can hold the peak and the two bins the criterion
compares; the C grid is the coarser one of the other notebooks (steps of 1 in log10, a deviation).

| Claim | Here |
|---|---|
| fig3b-decoding-time-course | tested, mice 2 and 3, with C chosen per bin |

In [ ]:
import time, resource, os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import najafi as nj

T0 = time.time()
A = nj.assets()
A = A[A.mouse.isin([2, 3])].reset_index(drop=True)
S = nj.load_all(A, workers=6, alignments=(nj.CHOICE,))
for s in S:
    s["al"][nj.CHOICE].pop("frames", None)
print("loaded", len(S), "sessions in %.0f s, peak memory %.1f GB" % (time.time() - T0, resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))

In [ ]:
D = pd.DataFrame([{"mouse": s["mouse"], "session": s["session_id"], "neurons": len(s["type"]),
                   "correct_low": int((s["trials"].choice.to_numpy()[nj.trial_mask(s, (1,))] == 0).sum()),
                   "correct_high": int((s["trials"].choice.to_numpy()[nj.trial_mask(s, (1,))] == 1).sum()),
                   "bins": f"{s['al'][nj.CHOICE]['k'][0]}..{s['al'][nj.CHOICE]['k'][-1]}"} for s in S])
print("sessions per mouse:", D.groupby("mouse").size().to_dict())
print("sessions with at least 10 correct trials of each choice:", D[(D.correct_low >= 10) & (D.correct_high >= 10)].groupby("mouse").size().to_dict())
print("choice-aligned bins per session:", D.bins.value_counts().to_dict())
print(D.groupby("mouse")[["neurons", "correct_low", "correct_high"]].median())

In [ ]:
def perbin_session(s, seed):
    rng = np.random.default_rng(seed)
    xy = nj._xy(s, (1,), rng=rng)
    if xy is None:
        return None
    Z = nj.zscore_bins(xy[0]); y = xy[1]
    ks = s["al"][nj.CHOICE]["k"]
    bins = [0] + [i for i, k in enumerate(ks) if k >= -3]
    acc, sh = nj.timecourse_perbin(Z, y, rng, bins, nrep=50)
    return {"mouse": s["mouse"], "session": s["session_id"], "k": ks[bins], "acc": acc, "sh": sh}

t1 = time.time()
OUT = Parallel(n_jobs=4)(delayed(perbin_session)(s, 9000 + i) for i, s in enumerate(S))
print("decoders: %.1f min; sessions decoded: %d of %d" % ((time.time() - t1) / 60, sum(o is not None for o in OUT), len(OUT)))

## fig3b-decoding-time-course

> Accuracy of the all-neuron choice decoder grows after stimulus onset and peaks at the time of
> the choice.

In [ ]:
TC = pd.DataFrame([{"mouse": o["mouse"], "k": int(k), "first": j == 0, "acc": a, "sh": b}
                   for o in OUT if o is not None for j, (k, a, b) in enumerate(zip(o["k"], o["acc"], o["sh"]))])
for m, g in TC.groupby("mouse"):
    first = g[g["first"]]
    late = g[~g["first"]]
    n = late.groupby("k").size(); nmax = n.max()
    v = late.groupby("k").acc.mean()[n >= 0.5 * nmax]; vs = late.groupby("k").sh.mean()[n >= 0.5 * nmax]
    kp = int(v.idxmax())
    second = v.drop(kp).max()
    print(f"mouse {m} ({nmax} sessions): earliest bin (k {first.k.min()} to {first.k.max()}) {first.acc.mean():.1f}, bin -1 {v.loc[-1]:.1f}, bin 0 {v.loc[0]:.1f}; "
          f"peak {v.max():.1f} at bin {kp} (next highest {second:.1f}); (a) peak in bins -1 or 0: {kp in (-1, 0)}; (b) peak within 300 ms (bins -3 to 2): {-3 <= kp <= 2}; "
          f"earliest below bin -1: {first.acc.mean() < v.loc[-1]}; shuffle {vs.min():.1f} to {vs.max():.1f}")
    print("   accuracy by bin:", v.round(2).to_dict())
    print("   sessions per bin:", n.to_dict())

**Result: not reproduced.** With C chosen at bin -1 and 25 splits (03), all-neuron accuracy on correct trials rises from the earliest bin (59.4 to 70.2 percent) to the bin before the choice (72.4 to 86.4) in every mouse; the peak falls at bin 2 in mice 1 and 4, bin 3 in mouse 2 (86.6, equal to bin 2 at one decimal) and bin 6 in mouse 3. With C chosen at every bin and 50 splits (04, mice 2 and 3), mouse 2 peaks at bin 2 (88.4, next 88.2) and mouse 3 at bin 6 (83.6). The peak is never in bins -1 or 0, reading (a), and in mouse 3 it lies outside 300 ms, reading (b), under both C rules, so the claim fails under both readings; the criterion has no partial band.


## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| data-composition | reproduced |
| fig1b-psychometric | reproduced |
| figs1b-trial-history | blocked, data missing |
| fig1gi-inferred-spiking-differs | not reproduced |
| fig2a-example-session | not reproduced |
| fig2bc-fraction-selective-similar | not reproduced |
| fig2bd-selectivity-time-course | blocked, methods underspecified |
| fig2de-selectivity-magnitude-similar | partially reproduced |
| figs3d-same-spiking-selectivity | reproduced |
| figs3c-wider-window | not reproduced |
| figs3e-correct-error-correlation | not reproduced |
| fig3bc-decoding-similar | reproduced |
| fig3b-decoding-time-course | not reproduced |
| fig3df-weights-similar | partially reproduced |
| fig3hi-mixed-no-advantage | reproduced |
| figs5-fluorescence-decoding | blocked, data missing |
| figs6ab-previous-choice-stimulus | blocked, other |
| figs6c-outcome-decoding | blocked, other |
| figs6de-incorrect-trials | not reproduced |
| fig4bd-stability-similar | reproduced |
| figs7c-angle-stability | blocked, too large |
| fig5a-nonselective-model | blocked, other |
| fig5bc-selective-models-match | blocked, other |
| figs9-signal-selective-best | blocked, other |
| fig6a-same-selectivity-correlations | reproduced |
| fig6bc-inhibitory-correlations-higher | blocked, methods underspecified |
| fig7-noise-correlations-limit-decoding | blocked, methods underspecified |
| fig8ab-learning-accuracy | not reproduced |
| fig8ce-choice-signal-onset | not reproduced |
| fig8de-learning-parallel | not reproduced |
| figs13c-equal-trials | not reproduced |
| figs12-licking-running | not reproduced |
| fig8f-correlations-decrease | partially reproduced |
| figs13ab-pseudo-learning | not reproduced |
| fig8g-fraction-selective-triples | not reproduced |
| figs2-temporal-dynamics-similar | blocked, other |
| figs14-neuropil | blocked, data missing |
| figs15-labeling-specificity | blocked, data missing |

See the attempt record `attempts/najafi-2020-01.yaml` for every claim's outcome.

In [ ]:
print("runtime %.1f min, peak memory %.1f GB" % ((time.time() - T0) / 60, resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))